# 12 — Product evals, annotation quality, safety checks, monitoring, and AI PM scorecards

## Mental model

A serious NLP product is not just a model. It is:

```text
data → model → evaluation → monitoring → feedback loop → product decision
```

This notebook gives you reusable evaluation and governance utilities.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import re
import json
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
df.head()

## Cohen's kappa for annotation agreement

Kappa measures agreement beyond chance.

```text
κ = (Pa - Pe) / (1 - Pe)
```

In [ ]:
def cohens_kappa(a, b):
    assert len(a) == len(b)
    labels = sorted(set(a) | set(b))
    n = len(a)
    observed = sum(x == y for x, y in zip(a, b)) / n
    pa = observed
    counts_a = {label: a.count(label) for label in labels}
    counts_b = {label: b.count(label) for label in labels}
    pe = sum((counts_a[l] / n) * (counts_b[l] / n) for l in labels)
    return (pa - pe) / (1 - pe) if pe < 1 else 1.0

ann1 = [1, 1, 0, 0, 0]
ann2 = [1, 0, 0, 0, 1]
print(cohens_kappa(ann1, ann2))

## Annotation guideline template

Use this before labelling data. If humans cannot agree, your model will not learn reliably.

In [ ]:
annotation_guidelines = {
    "task": "customer ticket sentiment",
    "labels": {
        "positive": "praise, satisfaction, clear approval",
        "negative": "complaint, failure, dissatisfaction, churn/cancel request",
        "neutral": "question, request, factual statement without clear affect",
    },
    "edge_cases": {
        "mixed_feedback": "use neutral unless negative risk is operationally important",
        "sarcasm": "label intended meaning, not literal wording",
        "billing/refund": "usually negative unless phrased as resolved praise",
    },
    "quality_checks": [
        "double-label 10-20% of data",
        "review low-agreement examples",
        "update guidelines before scaling annotation",
    ],
}
print(json.dumps(annotation_guidelines, indent=2))

## Build an evaluation set by buckets

A single random test set is not enough. Include normal, edge, adversarial, and high-stakes cases.

In [ ]:
eval_cases = pd.DataFrame([
    {"bucket": "easy", "text": "I love the product", "expected": "positive"},
    {"bucket": "easy", "text": "The product is terrible", "expected": "negative"},
    {"bucket": "mixed", "text": "The web app is great but the mobile app is slow", "expected": "neutral"},
    {"bucket": "sarcasm", "text": "Great, another crash right before my demo", "expected": "negative"},
    {"bucket": "privacy", "text": "What data do you store from my Google Drive?", "expected": "neutral"},
    {"bucket": "high_stakes", "text": "The AI invented a legal clause in my contract summary", "expected": "negative"},
])
eval_cases

## Output validation for structured LLM results

For company workflows, force structured output and validate it before taking action.

In [ ]:
def validate_extraction(obj, required_schema):
    errors = []
    if not isinstance(obj, dict):
        return ["output is not a dictionary"]
    for key, typ in required_schema.items():
        if key not in obj:
            errors.append(f"missing key: {key}")
        elif not isinstance(obj[key], typ):
            errors.append(f"wrong type for {key}: expected {typ.__name__}")
    return errors

schema = {"label": str, "confidence": float, "rationale": str}
outputs = [
    {"label": "negative", "confidence": 0.82, "rationale": "complaint about crash"},
    {"label": "negative", "confidence": "high"},
]
for o in outputs:
    print(o, validate_extraction(o, schema))

## PII redaction baseline

Use deterministic redaction before logging or sending data to external systems. This is only a baseline; production privacy needs deeper review.

In [ ]:
def redact_pii(text):
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", "[EMAIL]", text)
    text = re.sub(r"\b(?:\+?\d[\d\s().-]{7,}\d)\b", "[PHONE]", text)
    text = re.sub(r"\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b", "[CARD]", text)
    return text

sample = "Email priya@example.com or call +61 400 123 456. Card 4111 1111 1111 1111."
redact_pii(sample)

## Hallucination/factuality guardrails

A simple check: any number in the answer should appear in retrieved evidence. Extend this to dates, names, citations, and table values.

In [ ]:
def extract_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text))

def unsupported_numbers(answer, evidence):
    ev = set().union(*(extract_numbers(e) for e in evidence)) if evidence else set()
    return sorted(extract_numbers(answer) - ev)

evidence = ["The invoice upload bug affects files over 20MB."]
print(unsupported_numbers("The bug affects files over 20MB.", evidence))
print(unsupported_numbers("The bug affects files over 50MB.", evidence))

## Product scorecard

Use this to compare models, prompts, retrieval settings, or fine-tuned versions.

In [ ]:
scorecard = pd.DataFrame([
    {"metric": "task_macro_f1", "target": ">= 0.85", "current": None, "owner": "ML"},
    {"metric": "negative_recall", "target": ">= 0.90", "current": None, "owner": "PM/ML"},
    {"metric": "format_validity", "target": ">= 0.99", "current": None, "owner": "Eng"},
    {"metric": "unsupported_claim_rate", "target": "<= 0.02", "current": None, "owner": "ML"},
    {"metric": "p95_latency_seconds", "target": "<= 3.0", "current": None, "owner": "Eng"},
    {"metric": "cost_per_1000_requests", "target": "business-defined", "current": None, "owner": "Founder"},
    {"metric": "human_review_rate", "target": "business-defined", "current": None, "owner": "Ops"},
])
scorecard

## Monitoring log schema

Log enough to improve the system without violating privacy.

In [ ]:
monitoring_event = {
    "request_id": "uuid",
    "timestamp": "2026-06-16T10:00:00+10:00",
    "user_segment": "enterprise_admin",
    "input_hash": "sha256(...)",
    "redacted_input": "The app crashes when uploading [FILE]",
    "model_version": "sentiment-v3",
    "retrieval_doc_ids": [12, 19, 44],
    "output": {"label": "negative", "confidence": 0.91},
    "validation_errors": [],
    "latency_ms": 850,
    "cost_usd": 0.002,
    "user_feedback": None,
}
monitoring_event

## Founder checklist

Before shipping:

- define the user decision affected by the NLP output
- choose precision/recall trade-off intentionally
- build an eval set with edge cases
- validate output format
- redact sensitive data
- log failures and user feedback
- add human review for high-stakes cases
- document limitations in a model card